# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AakashDubba/aakash-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This data contract establishes the formal specification for **Lane 2: Refresh / Content Opportunity Scoring**.
It defines the unit of analysis, the operational decision moment, exact time windows, feature availability rules,
and verifies every data assertion with reproducible DuckDB queries against the FlyRank internship warehouse.

> **Lane Selection:** Lane 2 — Refresh / Content Opportunity Scoring  
> **Dataset:** FlyRank Internship Warehouse (`hf://datasets/FlyRank/internship-warehouse`)  
> **Development Partition:** March 2026 (`month=2026-03`)  

## 1. Unit of analysis + time window

### Plain-English Data Contract Specification (Questions 1–3)

#### 1. What does one row represent for my lane?
For **Lane 2 (Refresh / Content Opportunity Scoring)**, one row represents **one pseudonymized content item (`content_hash_id`)** belonging to an active client (`client_hash_id`), evaluated at the decision moment (cutoff date) for editorial triage.
- At the warehouse storage layer, `fact_content_daily_performance` records performance at the daily grain: `(report_date, client_hash_id, content_hash_id)`.
- For the Lane 2 opportunity scoring contract, daily performance records over the observation window are aggregated to the content item level so that each row represents an individual published asset being evaluated for refresh prioritization.

#### 2. Which warehouse table(s) will I use?
We use four core warehouse tables:
1. **`fact_content_daily_performance`** (partitioned by month, using mid-panel partition `month=2026-03`): provides daily time-series metrics including Google Search Console impressions (`gsc_impressions`), clicks (`gsc_clicks`), average ranking position (`gsc_avg_position`), and GA4 session signals.
2. **`dim_clients`**: client-level metadata and tracking history (`client_hash_id`, `is_active`, `access_profile`, `gsc_data_start`, `ga4_data_start`, `has_gsc_access`, `has_ga4_access`).
3. **`dim_content`**: static content metadata (`content_hash_id`, `content_type`, `search_volume`, `word_count`, `is_published`, `is_deleted`).
4. **`fact_content_query_90d`**: 90-day search query footprint (`content_visible_query_count`, query concentration).

#### 3. What time window will I use?
We use the mid-panel month of **March 2026** (`2026-03-01` to `2026-03-31`, 31 days). In strict accordance with repository skills, we avoid the final month (`_sample`, June 2026) to prevent developing inside the test window.

We define a clean temporal split around a realistic decision moment:
- **Decision Moment:** Midnight on **March 15, 2026** (the moment the content strategist runs the triage model).
- **Observation / Pre-period Feature Window:** **2026-03-01 to 2026-03-15** (15 days). All features must be strictly knowable on or before March 15.
- **Outcome / Post-period Evaluation Window:** **2026-03-16 to 2026-03-31** (16 days). Used solely to measure whether the page sustained traffic or suffered significant decay.

In [1]:
# Section 1 Code: Environment Setup, Secure Authentication & Warehouse Connectivity
import os, sys, winreg, time
import duckdb
import pandas as pd
import numpy as np

# Secure HF_TOKEN retrieval (env var -> Colab Secret -> Windows Registry)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    try:
        key = winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Environment')
        HF_TOKEN, _ = winreg.QueryValueEx(key, 'HF_TOKEN')
        os.environ['HF_TOKEN'] = HF_TOKEN
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Check for local cached snapshots (fast local execution without rate limits)
cache_dir = os.path.expanduser('~/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots')
if os.path.exists(cache_dir) and len(os.listdir(cache_dir)) > 0:
    snapshot = sorted(os.listdir(cache_dir))[-1]
    base = os.path.join(cache_dir, snapshot).replace('\\', '/')
    TABLES = {
        'dim_clients':       f"read_parquet('{base}/dim_clients.parquet')",
        'dim_content':       f"read_parquet('{base}/dim_content.parquet')",
        'fact_daily_march':  f"read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')",
        'fact_query_90d':    f"read_parquet('{base}/fact_content_query_90d.parquet')",
    }
else:
    TABLES = {
        'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
        'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
        'fact_daily_march':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
        'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    }

print("=== Warehouse Tables Verification ===")
for name, src in TABLES.items():
    n = con.execute(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:20} : {n:>12,} rows")

=== Warehouse Tables Verification ===
dim_clients          :          104 rows
dim_content          :      519,606 rows
fact_daily_march     :    9,841,378 rows
fact_query_90d       :    2,414,248 rows


## 2. Fields: feature / label / context / excluded

### Plain-English Data Contract Specification (Questions 4–5)

#### 4. What label or proxy will I predict/rank?
We predict **`is_declining`**, a binary classification label identifying content pieces experiencing search visibility decay in the outcome period.
- **Cohort Definition:** Active, published content items with measurable pre-period search visibility (`imp_pre >= 50` impressions in the first 15 days of March).
- **Mathematical Definition:**
  $$\text{is\_declining} = \begin{cases} 1 & \text{if } \text{imp\_post} < 0.80 \times \text{imp\_pre} \\ 0 & \text{otherwise} \end{cases}$$
- **Operational Meaning:** Pages that lose more than 20% of their organic search impressions between the first and second half of March 2026 represent immediate refresh opportunities where content expansion or re-optimization can halt further loss.

#### 5. What information will I deliberately exclude, and why?
1. **`imp_post`, `clk_post`, `avg_pos_post` (Outcome-period performance):**
   *Why excluded:* Future information. These metrics are recorded between March 16 and March 31, 2026. At the decision moment (March 15), they have not occurred. Including them causes severe target leakage.
2. **`trend_pct` and `trend_direction`:**
   *Why excluded:* Label-derived columns. In the FlyRank schema, trend direction is mathematically calculated from post-outcome performance. Using it in the feature matrix produces circular prediction.
3. **`client_hash_id`, `content_hash_id`, `url_hash_id`:**
   *Why excluded:* Identifiers. These are pseudonymized primary/foreign keys used solely for joins, client-grouped validation splitting, and triage identification. Allowing tree algorithms to split on IDs causes spurious memorization.
4. **`is_deleted = TRUE`:**
   *Why excluded:* Unactionable. Deleted/pruned pages are permanently removed from the website and cannot receive editorial refresh resources.

### Field Classification Table

| Field Name | Bucket | Source Table | Description & Rationale |
|---|---|---|---|
| `imp_pre` | **Feature** | `fact_content_daily_performance` | Pre-period (Mar 1–15) cumulative GSC impressions (demand exposure). |
| `clk_pre` | **Feature** | `fact_content_daily_performance` | Pre-period (Mar 1–15) cumulative GSC clicks (pre-decision organic traffic). |
| `avg_pos_pre` | **Feature** | `fact_content_daily_performance` | Pre-period average ranking position (ranking strength before cutoff). |
| `visible_queries` | **Feature** | `fact_content_query_90d` | Number of distinct search queries for which content ranks. |
| `word_count` | **Feature** | `dim_content` | Total word count of the published article (depth indicator). |
| `is_declining` | **Label** | Computed | Target proxy: 1 if `imp_post < 0.8 * imp_pre`, 0 otherwise. |
| `client_hash_id` | **Context** | `dim_clients` | Client pseudonym for grouped cross-validation and grouping. |
| `content_hash_id` | **Context** | `dim_content` | Unique row key for tracking triage recommendations. |
| `report_date` | **Context** | `fact_content_daily_performance` | Observation calendar date for temporal filtering. |
| `imp_post` | **Excluded** | `fact_content_daily_performance` | Future post-decision metric (strictly post-outcome). |
| `trend_pct` | **Excluded** | `fact_content_daily_performance` | Direct target-derived leakage. |
| `is_deleted` | **Excluded** | `dim_content` | Pruned content ineligible for editorial triage. |

In [2]:
# Section 2 Code: Build Pre/Post Cohort Frame and Verify Field Classifications
contract_cohort_sql = f"""
WITH daily_summary AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_pre,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS clk_pre,
        AVG(CASE WHEN report_date <= '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_pos_pre,
        -- OUTCOME METRIC (March 16 - 31, 2026)
        SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_post
    FROM {TABLES['fact_daily_march']}
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_pre >= 50
),
query_summary AS (
    SELECT 
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
),
content_summary AS (
    SELECT 
        content_hash_id,
        ANY_VALUE(word_count)   AS word_count,
        ANY_VALUE(is_published) AS is_published,
        ANY_VALUE(is_deleted)   AS is_deleted
    FROM {TABLES['dim_content']}
    GROUP BY content_hash_id
)
SELECT 
    d.client_hash_id,
    d.content_hash_id,
    d.imp_pre,
    d.clk_pre,
    ROUND(COALESCE(d.avg_pos_pre, 0.0), 2) AS avg_pos_pre,
    COALESCE(q.visible_queries, 0)         AS visible_queries,
    COALESCE(c.word_count, 0)              AS word_count,
    d.imp_post
FROM daily_summary d
LEFT JOIN query_summary q   ON d.content_hash_id = q.content_hash_id
LEFT JOIN content_summary c ON d.content_hash_id = c.content_hash_id
WHERE COALESCE(c.is_deleted, FALSE) IS FALSE
"""

df_cohort = con.execute(contract_cohort_sql).fetchdf()
df_cohort['is_declining'] = (df_cohort['imp_post'] < 0.80 * df_cohort['imp_pre']).astype(int)

print(f"Target Cohort Size (March 2026, imp_pre >= 50): {len(df_cohort):,} content items")
print("Target Distribution (is_declining):")
dist = df_cohort['is_declining'].value_counts(normalize=True).rename({0: 'Stable/Growing (0)', 1: 'Declining/Opportunity (1)'})
for label, pct in dist.items():
    count = (df_cohort['is_declining'] == (1 if '1' in label else 0)).sum()
    print(f"  {label:28}: {count:>6,} ({pct*100:.2f}%)")

print("\nCohort Feature Summary (Top 5 rows):")
display_cols = ['client_hash_id', 'content_hash_id', 'imp_pre', 'clk_pre', 'avg_pos_pre', 'visible_queries', 'word_count', 'is_declining']
print(df_cohort[display_cols].head().to_string(index=False))

Target Cohort Size (March 2026, imp_pre >= 50): 92,516 content items
Target Distribution (is_declining):
  Stable/Growing (0)          : 66,024 (71.36%)
  Declining/Opportunity (1)   : 26,492 (28.64%)

Cohort Feature Summary (Top 5 rows):
         client_hash_id          content_hash_id  imp_pre  clk_pre  avg_pos_pre  visible_queries  word_count  is_declining
client_400c21c81c8b46ef content_94148e3bea01e0c7    107.0      1.0        10.16                0        1536             0
client_400c21c81c8b46ef content_20951db0d80df60a     92.0      0.0         8.46                0        1376             0
client_400c21c81c8b46ef content_4deec3e08daa108d     74.0      0.0         8.28                2        1775             0
client_400c21c81c8b46ef content_cfe645fee7fbcf84     54.0      0.0         6.78                0        1827             1
client_400c21c81c8b46ef content_5c70731d94bc9b83    111.0      0.0         8.09                0        1636             1


## 3. Verify it with queries (grain, counts, missing values, windows)

### Exactly Three Verification Queries
We verify our contract assertions using three explicit queries executed against the real March 2026 warehouse partition:
1. **Grain Verification:** Prove that the warehouse table matches the stated unit/grain `(report_date, client_hash_id, content_hash_id)` with zero duplicates.
2. **Slice Size and Dates:** Quantify total row volume (9.84M rows), distinct client and content counts, and exact date span (`2026-03-01` to `2026-03-31`).
3. **Availability Filter (`IS TRUE`):** Filter using `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE` to verify how many rows survive.

---

### Five Features Maximum (Availability Statements)
Our model is strictly restricted to **five legitimate features**, each paired with its decision-moment availability rationale:

1. **`imp_pre`**: *Available when? This feature is knowable at the decision moment because it aggregates only historical search impressions recorded on or before March 15, 2026.*
2. **`clk_pre`**: *Available when? This feature is knowable at the decision moment because it measures historical organic click traffic captured in Google Search Console prior to the triage cutoff.*
3. **`avg_pos_pre`**: *Available when? This feature is knowable at the decision moment because average search ranking positions were logged during the 15-day observation window before editorial review.*
4. **`visible_queries`**: *Available when? This feature is knowable at the decision moment because the 90-day query footprint was already logged in search console inventory prior to the decision moment.*
5. **`word_count`**: *Available when? This feature is knowable at the decision moment because the published article length is a static attribute established in the CMS at publication time before March 2026.*

---

### Controlled Leakage Demonstration
To prove the necessity of the data contract, we conduct a controlled experiment:
- **Leaked Feature:** `imp_post` (the future impressions recorded between March 16 and March 31, 2026).
- **Why it leaks:** Because `is_declining` is defined as `imp_post < 0.8 * imp_pre`, including `imp_post` injects the outcome directly into the predictor.
- **Workflow:** We train a classifier with the leaked feature, measure its misleading score, remove `imp_post`, recompute the honest score, and explain why the inflated score is an illusion.

In [3]:
# Section 3 Code: The Three Verification Queries + Leakage vs. Honest Benchmark
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

print("===========================================================")
print("QUERY 1: Grain Verification (Duplicate Check)")
print("===========================================================")
q1_sql = f"""
SELECT 
    report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
FROM {TABLES['fact_daily_march']}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""
res1 = con.execute(q1_sql).fetchall()
print(f"Grain check duplicates found: {len(res1)}")
if len(res1) == 0:
    print("VERIFIED: The daily grain (report_date, client_hash_id, content_hash_id) is strictly unique with 0 duplicate rows.\n")

print("===========================================================")
print("QUERY 2: Slice Size and Dates (March 2026 Mid-Panel Partition)")
print("===========================================================")
q2_sql = f"""
SELECT 
    COUNT(*)                          AS total_daily_rows,
    COUNT(DISTINCT client_hash_id)    AS distinct_clients,
    COUNT(DISTINCT content_hash_id)   AS distinct_content_items,
    MIN(report_date)                  AS slice_start_date,
    MAX(report_date)                  AS slice_end_date
FROM {TABLES['fact_daily_march']};
"""
res2_df = con.execute(q2_sql).fetchdf()
print(res2_df.to_string(index=False))
print("VERIFIED: Slice contains exactly 9,841,378 daily records covering 2026-03-01 to 2026-03-31.\n")

print("===========================================================")
print("QUERY 3: Availability Filter (IS TRUE Verification)")
print("===========================================================")
q3_sql = f"""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_available_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) AS both_available_rows,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_both_surviving
FROM {TABLES['fact_daily_march']};
"""
res3_df = con.execute(q3_sql).fetchdf()
print(res3_df.to_string(index=False))
print("VERIFIED: Only 364,347 rows (3.70%) have both GSC and GA4 available simultaneously.\n")

print("===========================================================")
print("CONTROLLED LEAKAGE EXPERIMENT")
print("===========================================================")
honest_features = ['imp_pre', 'clk_pre', 'avg_pos_pre', 'visible_queries', 'word_count']
leaked_features = honest_features + ['imp_post']

# 75/25 Train-Test Split with stratified sampling
train_data, test_data = train_test_split(
    df_cohort, test_size=0.25, random_state=42, stratify=df_cohort['is_declining']
)

# 1. Model WITH Leaked Feature (imp_post)
clf_leaked = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
clf_leaked.fit(train_data[leaked_features], train_data['is_declining'])
y_prob_leaked = clf_leaked.predict_proba(test_data[leaked_features])[:, 1]
y_pred_leaked = clf_leaked.predict(test_data[leaked_features])
auc_leaked = roc_auc_score(test_data['is_declining'], y_prob_leaked)
f1_leaked  = f1_score(test_data['is_declining'], y_pred_leaked)
acc_leaked = accuracy_score(test_data['is_declining'], y_pred_leaked)

# 2. Honest Model (Leaked feature removed)
clf_honest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
clf_honest.fit(train_data[honest_features], train_data['is_declining'])
y_prob_honest = clf_honest.predict_proba(test_data[honest_features])[:, 1]
y_pred_honest = clf_honest.predict(test_data[honest_features])
auc_honest = roc_auc_score(test_data['is_declining'], y_prob_honest)
f1_honest  = f1_score(test_data['is_declining'], y_pred_honest)
acc_honest = accuracy_score(test_data['is_declining'], y_pred_honest)

comparison_df = pd.DataFrame([
    {'Configuration': 'Leaked Model (with imp_post)', 'ROC-AUC': round(auc_leaked, 4), 'F1-Score': round(f1_leaked, 4), 'Accuracy': round(acc_leaked, 4)},
    {'Configuration': 'Honest Model (5 safe features)', 'ROC-AUC': round(auc_honest, 4), 'F1-Score': round(f1_honest, 4), 'Accuracy': round(acc_honest, 4)}
])
print("Score Comparison Table:")
print(comparison_df.to_string(index=False))

print("\nClassification Report (Honest Model):")
print(classification_report(test_data['is_declining'], y_pred_honest, target_names=['Stable', 'Declining'], digits=4))

# Discard leaked feature to ensure final workflow retains only honest data
del clf_leaked, train_data, test_data
final_feature_matrix = df_cohort[honest_features]
print(f"\nRetained Final Feature Matrix Shape: {final_feature_matrix.shape} (strictly {len(honest_features)} features)")

QUERY 1: Grain Verification (Duplicate Check)


Grain check duplicates found: 0
VERIFIED: The daily grain (report_date, client_hash_id, content_hash_id) is strictly unique with 0 duplicate rows.

QUERY 2: Slice Size and Dates (March 2026 Mid-Panel Partition)
 total_daily_rows  distinct_clients  distinct_content_items slice_start_date slice_end_date
          9841378                55                  331437       2026-03-01     2026-03-31
VERIFIED: Slice contains exactly 9,841,378 daily records covering 2026-03-01 to 2026-03-31.

QUERY 3: Availability Filter (IS TRUE Verification)


 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows  pct_both_surviving
    9841378             3611061              413966               364347                 3.7
VERIFIED: Only 364,347 rows (3.70%) have both GSC and GA4 available simultaneously.

CONTROLLED LEAKAGE EXPERIMENT


Score Comparison Table:
                 Configuration  ROC-AUC  F1-Score  Accuracy
  Leaked Model (with imp_post)   0.9394    0.5539    0.8217
Honest Model (5 safe features)   0.7160    0.2059    0.7286

Classification Report (Honest Model):
              precision    recall  f1-score   support

      Stable     0.7341    0.9716    0.8363     16506
   Declining     0.6345    0.1229    0.2059      6623

    accuracy                         0.7286     23129
   macro avg     0.6843    0.5472    0.5211     23129
weighted avg     0.7056    0.7286    0.6558     23129


Retained Final Feature Matrix Shape: (92516, 5) (strictly 5 features)


## 4. Data limits

### Concrete Limitation of Chosen Data Slice

**Observed Limitation: Severe Cross-Platform Availability Bottleneck and Asymmetric History Start Dates**

1. **Measured Reality:** As proven by Query 3, out of 9,841,378 daily records in March 2026, **only 364,347 rows (3.70%)** have both `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`. Furthermore, GSC data is active on 36.69% of records while GA4 is active on only 4.21%.
2. **Root Cause in `dim_clients`:** Inspection of `dim_clients` reveals that client tracking start dates are heavily staggered. While Google Search Console was integrated across 2025 for many clients, Google Analytics 4 (GA4) was onboarded much later or not at all (`ga4_data_start` is NULL or late for over half the client portfolio).
3. **Practical Consequence for Modeling:** If a data contract naively requires an inner join on both search visibility (GSC) and user engagement/conversion metrics (GA4) across a uniform calendar slice, **it eliminates over 96% of the daily performance inventory and drops more than half of the client accounts** from opportunity evaluation. This creates severe survivor bias toward a tiny subset of mature, enterprise clients.
4. **Architectural Mitigation:** In production, Lane 2 triage contracts must decouple organic opportunity discovery (which relies on GSC ranking and query footprint) from post-click engagement audits (GA4). Search opportunity ranking should operate on GSC data for broad coverage, using GA4 engagement signals as an enrichment layer only when `ga4_data_available IS TRUE`.

In [4]:
# Section 4 Code: Quantify Client History Imbalance and Availability Bottleneck
client_history_sql = f"""
SELECT 
    COUNT(*)                                                               AS total_clients,
    COUNT(CASE WHEN gsc_data_start IS NOT NULL THEN 1 END)                 AS clients_with_gsc,
    COUNT(CASE WHEN ga4_data_start IS NOT NULL THEN 1 END)                 AS clients_with_ga4,
    COUNT(CASE WHEN gsc_data_start <= '2026-03-01' THEN 1 END)             AS gsc_ready_by_march_2026,
    COUNT(CASE WHEN ga4_data_start <= '2026-03-01' THEN 1 END)             AS ga4_ready_by_march_2026,
    COUNT(CASE WHEN gsc_data_start <= '2026-03-01' AND ga4_data_start <= '2026-03-01' THEN 1 END) AS both_ready_march_2026
FROM {TABLES['dim_clients']};
"""
client_limits_df = con.execute(client_history_sql).fetchdf()
print("=== Client History Readiness for March 2026 Slice ===")
print(client_limits_df.to_string(index=False))

coverage_sql = f"""
SELECT 
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS row_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_slice
FROM {TABLES['fact_daily_march']}
GROUP BY 1, 2, 3, 4
ORDER BY row_count DESC;
"""
coverage_df = con.execute(coverage_sql).fetchdf()
print("\n=== March 2026 Daily Row Availability Breakdown ===")
print(coverage_df.to_string(index=False))

=== Client History Readiness for March 2026 Slice ===
 total_clients  clients_with_gsc  clients_with_ga4  gsc_ready_by_march_2026  ga4_ready_by_march_2026  both_ready_march_2026
           104                67                51                       52                       26                     25

=== March 2026 Daily Row Availability Breakdown ===
 client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  row_count  pct_of_slice
           True            True               False               False    4690323         47.66
           True            True                True               False    1718348         17.46
           True           False                True                <NA>    1528366         15.53
           True           False               False                <NA>    1490375         15.14
           True            True                True                True     364347          3.70
           True            True               False            

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.